Notes: 

cell builder is currently embedded in the simulation module... Need to build the 

Also want to implement new clustering method...

In [1]:
import sys
sys.path.append('..')
sys.path.append('../Modules')

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
print(os.getcwd())
if 'notebook' in os.getcwd():
    # os.chdir("../scripts") # go from Neural-Modeling/notebooks to Neural-Modeling/scripts, where simulation outputs will be generated to. (maybe a separate folder could be used...)
    os.chdir("../simulations") # go to output folder
    print(os.getcwd())

/home/drfrbc/Neural-Modeling/notebooks
/home/drfrbc/Neural-Modeling/simulations


User Specifications -  get parameters, simulation folder

In [2]:
from Modules.constants import HayParameters
import datetime
import pickle
from neuron import h

sim_set_title = "description_of_simulation_set"
# create simulation folder
sims_dir = f"{datetime.datetime.now().strftime('%Y-%m-%d-%H-%M')}-{sim_set_title}"
os.makedirs(sims_dir, exist_ok=True)

# this is a quick example, but the generation of parameter sets and titles is in scripts/gen_param_list_advanced.py
# @TODO: implement generating these as seen in gen_param_list_advanced.py
sim_titles = ["description_of_simulation"]
parameter_sets = [HayParameters(sim_title, all_synapses_off=True) for sim_title in sim_titles] # example parameter set

# @TODO: compile modfiles once

# create simulation folders and save parameters in them
for parameters, sim_title in zip(parameter_sets, sim_titles):
    # create simulation folder
    sim_dir = os.path.join(sims_dir, sim_title)
    os.makedirs(sim_dir, exist_ok=True)

    with open(os.path.join(sim_dir, "parameters.pickle"), 'wb') as file:
        pickle.dump(parameters, file)

    # load modfiles
    try:
        h.load_file('stdrun.hoc')
        # h.nrn_load_dll('./x86_64/.libs/libnrnmech.so' # IF IN SCRIPTS FOLDER
        load_modfiles = h.nrn_load_dll('../scripts/x86_64/.libs/libnrnmech.so') # IF IN SIMULATIONS FOLDER
        if load_modfiles != 1:
            raise Exception("Error loading mod files")
        else:
            print("Mod files loaded successfully")
    except:
        # Already loaded
        pass 

# @TODO: at simulation time, the parameters should be loaded from each sim_dir in sims_dir. 
# We can make one script that does this code snippet, then passes sims_dir to the mpiexec simulation script.

--No graphics will be displayed.


Mod files loaded successfully


debugging sec_type_precise generation by viewing selected segments

In [3]:
# from Modules.cell_builder import CellBuilder
# from Modules.logger import Logger
# from Modules.cell_builder import SkeletonCell

# logger= Logger(sim_dir)

# # build the cell
# # logger.log(f"Building cell to generate segments.csv")
# cell_builder = CellBuilder(getattr(SkeletonCell, parameters.skeleton_cell_type), parameters, logger)
# cell, _ = cell_builder.build_cell()
# # logger.log(f"Cell built successfully.")

# # logger.log(f"Changing cell morphology, segmentation, etc")
# # manipulate morphology: reduction, segmentation
# #@TODO add code from cellbuilder.py: CellBuilder.build_cell  -lines around reductor code block
# ####
# ####
# # logger.log(f"Finished changing cell morphology, segmentation, etc")

# # logger.log("Saving adjacency matrix")
# if parameters.save_adj_matrix:
#     adj_matrix = cell.compute_directed_adjacency_matrix()
#     np.savetxt(os.path.join(parameters.path, "adj_matrix.txt"), adj_matrix)
# # logger.log("Finished saving adjacency matrix")

# # logger.log("Getting segments data")

# # save segments csv in simulation folder - the rest of this cell
# #@TODO: Make modularized code for this and clean. standardize between here and cell_model (this is from simulation.py)
# #@TODO: clean up cell.get_segments alongside cell.get_segments_of_type
# #@TODO: add sec_type (or another name for the variable) for denoting the segment type at the 'distal_basal' level instead of 'dend' for example.
# # Classify segments by morphology, save coordinates
# segments, seg_data = cell.get_segments(["all"]) # (segments is returned here to preserve NEURON references)
# seg_sections = []
# seg_idx = []
# seg_coords = []
# seg_half_seg_RAs = []
# seg = []
# seg_Ls = []
# sec_Ls = []
# sec_Ds = []
# seg_distance = []
# psegs=[]

# for i,entry in enumerate(seg_data):
#     # if parameters.build_stylized: #@DEPRACATED
#     #     sec_name = entry.section.split(".")[-1]
#     # else:
#     sec_name = entry.section.split(".")[-1] # name[idx]
#     #print(f"sec_name: {sec_name}")
#     seg_sections.append(sec_name.split("[")[0])
#     seg_idx.append(sec_name.split("[")[1].split("]")[0])
#     seg_coords.append(entry.coords)
#     seg_half_seg_RAs.append(entry.seg_half_seg_RA)
#     seg.append(entry.seg)
#     seg_Ls.append(entry.L)
#     psegs.append(entry.pseg)
#     sec_Ls.append(segments[i].sec.L)
#     sec_Ds.append(segments[i].sec.diam)
#     seg_distance.append(h.distance(segments[0], segments[i]))
    
    
# seg_sections = pd.DataFrame({ #@TODO: rename seg_sections to seg_sec_data or something
#     "section": seg_sections, 
#     "idx_in_section_type": seg_idx,
#     "seg_half_seg_RA": seg_half_seg_RAs,
#     "L": seg_Ls,
#     "seg":seg,
#     "pseg":psegs,
#     "Section_L":sec_Ls,
#     "Section_diam":sec_Ds,
#     "Distance":seg_distance,
#     })

# seg_coords = pd.concat(seg_coords)

# seg_data = pd.concat((seg_sections.reset_index(drop = True), seg_coords.reset_index(drop = True)), axis = 1) #@TODO: compute these together instead or make seg_sections computation more concise?
# seg_data = seg_data.reset_index(drop=True) #@TODO: check if this is needed
# seg_data['seg_id'] = seg_data.index # add a seg_id so that row i → seg_id i
# seg_data.to_csv(os.path.join(sim_dir, "segment_data1.csv"))
# # logger.log("Saved segments data to segment_data.csv")

# ### new version @TODO: finish this implementation for adding the new sec_types to segments.csv
# sec_types_to_get = np.unique([sec_type for syn_properties in [parameters.exc_syn_properties, parameters.inh_syn_properties] for sec_type in syn_properties.keys()])
# print(f"getting segments of types: {sec_types_to_get} for synapses")
# # These are the types the method handles currently:
# # sec_types_to_get = [
# #     'soma',
# #     'perisomatic',
# #     'trunk',
# #     'distal_basal',
# #     'distal_apic',
# #     'nexus',
# #     'tuft',
# #     'oblique'
# # ]
# rows = []
# for stype in sec_types_to_get:
#     try:
#         segs = cell.get_segments_of_type(stype)
#     except ValueError:
#         # in case a type is empty / not implemented
#         continue
#     for seg in segs:
#         rows.append({
#             'sec_name': seg.sec.name(),  # e.g. "/cell/apic[12]"
#             'seg_x':    seg.x,           # normalized position along the section
#             'sec_type': stype,
#             'seg_id': segments.index(seg), # index of the segment in the list returned from cell.get_segments(['all'])
#         })

# df = pd.DataFrame(rows, # @TODO: check this dataframe. remove duplicate segments. include segment id from the index of the list returned from cell.get_segments(['all'])
#         columns=['sec_name','seg_x','sec_type', 'seg_id'])
# df.to_csv(os.path.join(sim_dir, "segment_data2.csv"), index=False)
# ###

# ### combining main seg_data with this precise sec_type
# # print out any seg_ids that have more than one precise type #@TODO: debugging overlapping precise sec_types that may need clearer definitions or stricter logic
# # (e.g. "perisomatic" and "trunk")
# # group to collect all sec_type per seg_id
# grouped = df.groupby('seg_id')['sec_type'].unique()
# overlaps = grouped[grouped.apply(lambda arr: len(arr) > 1)]
# if not overlaps.empty:
#     print("Segments with multiple precise sec_types:")
#     for sid, types in overlaps.items():
#         print(f"  seg_id {sid}: {types.tolist()}")

# # 5) build a map: if there's exactly one type, keep it; otherwise None
# precise_map = grouped.apply(lambda arr: arr[0] if len(arr) == 1 else None)

# # 6) assign into your main DataFrame
# seg_data['sec_type_precise'] = seg_data['seg_id'].map(precise_map)

# # 7) write the integrated CSV back out (overwriting the old one)
# seg_data.to_csv(os.path.join(sim_dir, "segment_data.csv"), index=False)
# # logger.log("Saved integrated segment_data.csv with sec_type_precise")

In [4]:
# cell.get_segments_of_type("trunk")

In [5]:
# from Modules.plot_morphology import plot_reduced_morphology
# plot_reduced_morphology(seg_data, deleted_indices=[2573, 1111, 2300, 1250, 1359, 1476, 1758, 2479, 2284, 1395, 1548], radius_scale=1.1) # trunk stop segments

In [6]:
# # for i,j in overlaps.items():
# #     print(i)

# overlapping_seg_ids = [i for i,j in overlaps.items()]

In [7]:
# nexus_seg_ids = seg_data[seg_data['sec_type_precise'] == 'nexus']['seg_id'].tolist()
# tuft_seg_ids = seg_data[seg_data['sec_type_precise'] == 'tuft']['seg_id'].tolist()

In [8]:
# plot_reduced_morphology(seg_data,deleted_indices=overlapping_seg_ids)

In [9]:
# plot_reduced_morphology(seg_data,deleted_indices=nexus_seg_ids)

In [10]:
# plot_reduced_morphology(seg_data,deleted_indices=tuft_seg_ids)

In [11]:
# unlabeled_seg_ids = seg_data[seg_data['sec_type_precise'].isna()]['seg_id'].tolist()

In [12]:
# seg_data[seg_data['sec_type_precise'].isna()].iloc[:, [1, 3, 4, 8, 9,10,11,12,13,14,15,16,17,18,-1]]

In [13]:
# plot_reduced_morphology(seg_data,deleted_indices=unlabeled_seg_ids)

In [14]:
# plot_reduced_morphology(seg_data,deleted_indices=seg_data[seg_data['sec_type_precise'] == 'trunk']['seg_id'].tolist())

Build cell to save segments csv

In [15]:
from Modules.cell_builder import CellBuilder
from Modules.logger import Logger
from Modules.cell_builder import SkeletonCell
# this script would be called with mpiexec

# @TODO: would we want to use mpiexec for this script? It depends on if there are different morphologies: different cell types, different reduction parameters, and segmentation. Yes, one per simulation folder. might as well.
# Additionally, do we save these in simulation folders or try to use a common folder? A: simulation folder

# for rank in [0]: # replace with actual mpi implementation


def generate_segments_csv(sim_dir, parameters, logger): #@TODO: add this to some class... Maybe not CellBuilder because the CellBuilder instance could be temporary instead? discuss with @davidfague
    # build the cell
    logger.log(f"Building cell to generate segments.csv")
    cell_builder = CellBuilder(getattr(SkeletonCell, parameters.skeleton_cell_type), parameters, logger)
    cell, _ = cell_builder.build_cell()
    logger.log(f"Cell built successfully.")

    logger.log(f"Changing cell morphology, segmentation, etc")
    # manipulate morphology: reduction, segmentation
    #@TODO add code from cellbuilder.py: CellBuilder.build_cell  -lines around reductor code block
    ####
    ####
    logger.log(f"Finished changing cell morphology, segmentation, etc")

    logger.log("Saving adjacency matrix")
    if parameters.save_adj_matrix:
        adj_matrix = cell.compute_directed_adjacency_matrix()
        np.savetxt(os.path.join(parameters.path, "adj_matrix.txt"), adj_matrix)
    logger.log("Finished saving adjacency matrix")

    logger.log("Getting segments data")

    # save segments csv in simulation folder - the rest of this cell
    #@TODO: Make modularized code for this and clean. standardize between here and cell_model (this is from simulation.py)
    #@TODO: clean up cell.get_segments alongside cell.get_segments_of_type
    #@TODO: add sec_type (or another name for the variable) for denoting the segment type at the 'distal_basal' level instead of 'dend' for example.
    # Classify segments by morphology, save coordinates
    segments, seg_data = cell.get_segments(["all"]) # (segments is returned here to preserve NEURON references)
    seg_sections = []
    seg_idx = []
    seg_coords = []
    seg_half_seg_RAs = []
    seg = []
    seg_Ls = []
    sec_Ls = []
    sec_Ds = []
    seg_distance = []
    psegs=[]
    
    for i,entry in enumerate(seg_data):
        # if parameters.build_stylized: #@DEPRACATED
        #     sec_name = entry.section.split(".")[-1]
        # else:
        sec_name = entry.section.split(".")[-1] # name[idx]
        #print(f"sec_name: {sec_name}")
        seg_sections.append(sec_name.split("[")[0])
        seg_idx.append(sec_name.split("[")[1].split("]")[0])
        seg_coords.append(entry.coords)
        seg_half_seg_RAs.append(entry.seg_half_seg_RA)
        seg.append(entry.seg)
        seg_Ls.append(entry.L)
        psegs.append(entry.pseg)
        sec_Ls.append(segments[i].sec.L)
        sec_Ds.append(segments[i].sec.diam)
        seg_distance.append(h.distance(segments[0], segments[i]))
        
        
    seg_sections = pd.DataFrame({ #@TODO: rename seg_sections to seg_sec_data or something
        "section": seg_sections, 
        "idx_in_section_type": seg_idx,
        "seg_half_seg_RA": seg_half_seg_RAs,
        "L": seg_Ls,
        "length": seg_Ls,
        "seg":seg,
        "pseg":psegs,
        "Section_L":sec_Ls,
        "Section_diam":sec_Ds,
        "Distance":seg_distance,
        })

    seg_coords = pd.concat(seg_coords)

    seg_data = pd.concat((seg_sections.reset_index(drop = True), seg_coords.reset_index(drop = True)), axis = 1) #@TODO: compute these together instead or make seg_sections computation more concise?
    seg_data = seg_data.reset_index(drop=True) #@TODO: check if this is needed
    seg_data['seg_id'] = seg_data.index # add a seg_id so that row i → seg_id i
    seg_data.to_csv(os.path.join(sim_dir, "segment_data1.csv"))
    logger.log("Saved segments data to segment_data.csv")

    ### new version @TODO: finish this implementation for adding the new sec_types to segments.csv
    sec_types_to_get = np.unique([sec_type for syn_properties in [parameters.exc_syn_properties, parameters.inh_syn_properties] for sec_type in syn_properties.keys()])
    print(f"getting segments of types: {sec_types_to_get} for synapses")
    # These are the types the method handles currently:
    # sec_types_to_get = [
    #     'soma',
    #     'perisomatic',
    #     'trunk',
    #     'distal_basal',
    #     'distal_apic',
    #     'nexus',
    #     'tuft',
    #     'oblique'
    # ]
    rows = []
    for stype in sec_types_to_get:
        try:
            segs = cell.get_segments_of_type(stype)
        except ValueError:
            # in case a type is empty / not implemented
            continue
        for seg in segs:
            rows.append({
                'sec_name': seg.sec.name(),  # e.g. "/cell/apic[12]"
                'seg_x':    seg.x,           # normalized position along the section
                'sec_type': stype,
                'seg_id': segments.index(seg), # index of the segment in the list returned from cell.get_segments(['all'])
            })

    df = pd.DataFrame(rows, # @TODO: check this dataframe. remove duplicate segments. include segment id from the index of the list returned from cell.get_segments(['all'])
            columns=['sec_name','seg_x','sec_type', 'seg_id'])
    df.to_csv(os.path.join(sim_dir, "segment_data2.csv"), index=False)
    ###

    ### combining main seg_data with this precise sec_type
    # print out any seg_ids that have more than one precise type #@TODO: debugging overlapping precise sec_types that may need clearer definitions or stricter logic
    # (e.g. "perisomatic" and "trunk")
    # group to collect all sec_type per seg_id
    grouped = df.groupby('seg_id')['sec_type'].unique()
    overlaps = grouped[grouped.apply(lambda arr: len(arr) > 1)]
    if not overlaps.empty:
        print("Segments with multiple precise sec_types:")
        for sid, types in overlaps.items():
            print(f"  seg_id {sid}: {types.tolist()}")

    # 5) build a map: if there's exactly one type, keep it; otherwise None
    precise_map = grouped.apply(lambda arr: arr[0] if len(arr) == 1 else None)

    # 6) assign into your main DataFrame
    seg_data['sec_type_precise'] = seg_data['seg_id'].map(precise_map)
    seg_data.loc[seg_data['section'] == 'axon', 'sec_type_precise'] = 'axon'
    seg_data.loc[seg_data['section'] == 'soma', 'sec_type_precise'] = 'soma'

    # 7) write the integrated CSV back out (overwriting the old one)
    seg_data.to_csv(os.path.join(sim_dir, "segment_data.csv"), index=False)
    logger.log("Saved integrated segment_data.csv with sec_type_precise")


from collections.abc import Callable, Iterable
from typing import Mapping, Union

def run_on_all_sims( #TODO: Use MPI to process these in parallel. Evenly Distribute sims to workers instead of using 1 per rank. add to class?
    sims_dir: str,
    process_fns: Union[Callable[[str, Mapping, object], None],
                       Iterable[Callable[[str, Mapping, object], None]]]
) -> None:
    """process_fns can be a single function or a list of functions. 
    Each function should take the simulation directory, parameters, and logger as arguments."""
    # normalize to a list
    if callable(process_fns):
        fns = [process_fns]
    else:
        fns = list(process_fns)

    for entry in os.listdir(sims_dir):
        sim_dir = os.path.join(sims_dir, entry)
        if not os.path.isdir(sim_dir):
            continue

        # load parameters
        with open(os.path.join(sim_dir, "parameters.pickle"), "rb") as f:
            parameters = pickle.load(f)

        logger = Logger(sim_dir) # create per‑sim logger (write info into "sims_dir/sim_dir/log.txt")

        for fn in fns :# run each processing function
            fn(sim_dir=sim_dir, parameters=parameters, logger=logger)

# for sim_dir in os.listdir(sims_dir):
#     sim_dir = os.path.join(sims_dir, sim_dir)
#     if not os.path.isdir(sim_dir):
#         continue

#     # load parameters
#     with open(os.path.join(sim_dir, "params.pickle"), 'rb') as file:
#         parameters = pickle.load(file)

#     logger = Logger(sim_dir)

#     generate_segments_csv(sim_dir, parameters)

run_on_all_sims(sims_dir, generate_segments_csv) # generate the segments csv for each simulation

Removing duplicate coordinate at index 1 in section L5PCtemplate[0].apic[0]
getting segments of types: ['distal_basal' 'nexus' 'oblique' 'perisomatic' 'trunk' 'tuft'] for synapses


In [16]:
parameters.h_tstop

5000

Generate synapse Locations - load segments csv

In [17]:
# load segments csv

## generate synapse locations - code from cellbuilder @TODO: need to change to utilize segment_data.csv instead of cell_model.py

# save to synapses csv in simulation folder # example in cell_builder of getting properties

import pandas as pd
from functools import partial
# define a class for generating synapses abstractly
class PreSimSynapseGenerator:
    def __init__(self, segments, parameters, logger):
        self.segments = segments
        self.parameters = parameters
        self.synapses = pd.DataFrame()
        self.logger = logger

        if self.parameters.segment_measurement_for_probabilities not in ['length', 'surface_area']:
            raise ValueError(f"Measurement for probabilities must be 'length' or 'surface_area'. Not {self.parameters.segment_measurement_for_probabilities}.")

    def generate_synapse_locations(self):
        for syn_properties_set, use_density, syn_mod, syn_params_choices in zip([self.parameters.exc_syn_properties, self.parameters.inh_syn_properties],
                                                            [self.parameters.exc_use_density, self.parameters.inh_use_density],
                                                            [self.parameters.exc_syn_mod, self.parameters.inh_syn_mod],
                                                            [self.parameters.exc_syn_params_choices, self.parameters.inh_syn_params_choices]):
            for sec_type, synapse_properties in syn_properties_set.items():
                self.logger.log(f"Generating synapses for {sec_type.upper()} with properties: {synapse_properties}")
                segments_to_generate_on = self.get_segments_of_type(sec_type, self.segments) # get segments
                self.random_state = np.random.RandomState(self.parameters.inh_syn_properties[sec_type]['seed']['synapses'])
                np.random.seed(self.parameters.inh_syn_properties[sec_type]['seed']['synapses'])

                synapses_this_sec_type = self.build_synapses_with_specs(segments_to_generate_on = segments_to_generate_on,
                                        sec_type = sec_type,
                                        synapse_type = synapse_properties['synapse_type'],
                                        use_density = use_density,
                                        syn_number = synapse_properties['syn_number'] if not use_density else None,
                                        syn_density = synapse_properties['syn_density'] if use_density else None,
                                        initial_weight_distribution = synapse_properties['initial_weight_distribution'],
                                        release_probability_distribution = synapse_properties['release_probability_distribution'],
                                        syn_mod = syn_mod,
                                        syn_params_choices = syn_params_choices,
                                        name = f"{synapse_properties['synapse_type']}_{sec_type}"
                                        )
                self.synapses = pd.concat((self.synapses, synapses_this_sec_type), ignore_index=True)
        self.synapses = self.synapses.reset_index(drop=True) #@TODO: check if this is necessary and check self.synapses.

    def get_segments_of_type(self, sec_type, segments):
        return segments[segments['sec_type_precise'] == sec_type]
    
    def build_synapses_with_specs(self, segments_to_generate_on: pd.DataFrame, sec_type: str, synapse_type: str, use_density: bool,
                                  syn_number: int, syn_density: float, initial_weight_distribution: dict,
                                  release_probability_distribution: dict, name: str, syn_mod:str, syn_params_choices) -> pd.DataFrame:
        
        if initial_weight_distribution is None or release_probability_distribution is None:
            raise ValueError("Both gmax_dist_params and P_release_params must be provided.")
        
        if syn_number is None and syn_density is None:
            raise ValueError("Either syn_number or syn_density must be provided.")
        elif syn_number is not None and syn_density is not None:
            raise ValueError("Only one of syn_number or syn_density must be provided.")
        elif syn_number is not None and use_density:
            raise ValueError("syn_number should be none when using density.")
        elif syn_density is not None and not use_density:
            raise ValueError("syn_density should be none when not using density.")
        
        # if len(np.unique(segments_to_generate_on)) != len(segments_to_generate_on):
        #     raise ValueError(f"Segments to generate on must be unique. Not {segments_to_generate_on}.")

        #@TODO: check that this doesn't throw an error with expected inputs.
        #@TODO: move initial_weight_distribution info to a dictionary within synapse_properties, same for release probability_distribution.
        if not isinstance(syn_params_choices, dict):
            raise ValueError(f"syn_params_choices must be a dict. Not {type(syn_params_choices)}. syn_params_choices: {syn_params_choices}.")
        # if False in [True if isinstance(syn_params, dict) else False for syn_params in syn_params_choices]: # changed to dict with keys 'choices' and 'probs' where each choice is a dict.
        #     raise ValueError(f"syn_params_choices must be a dict of dictionaries. Not {type(syn_params_choices)}. syn_params_choices: {syn_params_choices}.")

        if initial_weight_distribution['function'] is not None:
            initial_weight_distribution = partial(initial_weight_distribution['function'], **initial_weight_distribution['params'], size=1)
            initial_weight_distribution_is_partial = True
        else:
            initial_weight_distribution = initial_weight_distribution['params']['mean'] # use mean if no function is provided
            initial_weight_distribution_is_partial = False
        self.logger.log(f"Initial weight distribution for {name}: {initial_weight_distribution}")

        if release_probability_distribution['function'] is not None:
            release_probability_distribution = partial(release_probability_distribution['function'], **release_probability_distribution['params'], size=1)
            release_probability_distribution_is_partial = True
        else:
            release_probability_distribution = release_probability_distribution['params']['mean']
            release_probability_distribution_is_partial = False
        self.logger.log(f"Release probability distribution for {name}: {release_probability_distribution}")

        # calculate probabilities of placing synapses on segments
        total_measurement = segments_to_generate_on[self.parameters.segment_measurement_for_probabilities].sum()
        self.logger.log(f"Total {self.parameters.segment_measurement_for_probabilities} for {name}: {total_measurement}")
        # segments_to_generate_on['probability'] = segments_to_generate_on[self.parameters.segment_measurement_for_probabilities] / total_measurement
        segments_to_generate_on = segments_to_generate_on.assign(
            probability=lambda df: df[self.parameters.segment_measurement_for_probabilities] 
                                / total_measurement
        )

        if segments_to_generate_on['probability'].sum() < 0.999 or segments_to_generate_on['probability'].sum() > 1.001:
            raise ValueError(f"Probabilities do not sum to 1 instead {segments_to_generate_on['probability'].sum()}. Check your segment measurement for probabilities.")

        if use_density:
            # calculate number of synapses per segment
            syn_number = int(total_measurement * syn_density) #@TODO make compatible with syn_density being function instead of float (not at all urgent)
        self.logger.log(f"Number of synapses being generated for {name}: {syn_number}")

        # synapses = pd.DataFrame(columns=['name', 'modfile', 'initW', 'gmax', 'release_probability', 'seg_id']) #@TODO: add columns for possible syn_params keys

        for _ in range(syn_number): # @TODO: do this in parallel instead of serial. Use list comprehension?
            # sample a segment
            segment_id = self.random_state.choice(a=segments_to_generate_on['seg_id'], size=1, replace=True, p=segments_to_generate_on['probability'])[0] #@TODO: check if [0] is necessary
            segment = segments_to_generate_on[segments_to_generate_on['seg_id'] == segment_id]
            # choose sub-synapse type (short term plasticity, gbar, etc. properties)
            if len(syn_params_choices['choices']) == 1:
                syn_params_made_choice = syn_params_choices['choices'][0]
            elif syn_params_choices['probs'] == 'perisomatic_distance':
                syn_params_made_choice = syn_params_choices['choices'][1] if segment['Distance'].values[0] > 100 else syn_params_choices['choices'][0]
            else:
                syn_params_made_choice = self.random_state.choice(syn_params_choices['choices'], p=syn_params_choices['probs']) # choose between CS2CP and CP2CP if it is AMPA. pyr2pyr will not be a tuple or list.
            # print(syn_params_choices)
            # print(syn_params_made_choice)
            # print(syn_params_made_choice.items())
            # choice_name, syn_params_this_syn = syn_params_made_choice.items()
            # syn_params_this_syn['syn_params_choice'] = choice_name
            choice_name, syn_params_this_syn = next(iter(syn_params_made_choice.items())) #TODO: check with choices in constants.__post_init__
            syn_params_this_syn['syn_params_choice'] = choice_name

            #TODO: update synapses.py so that syn_params that use AMPA_NMDA and GABA_AB modfiles have modfile indicated.

            # sample a release probability
            if release_probability_distribution_is_partial:
                syn_params_this_syn["release_probability"] = release_probability_distribution(size=1) # sample distribution
            else:
                syn_params_this_syn["release_probability"] = release_probability_distribution # distribution is a constant
            if 'int2pyr' in syn_mod or 'pyr2pyr' in syn_mod:  # these modfiles do release probability computation as spikes arrive during simulation instead of before
                syn_params_this_syn["P_0"] = syn_params_this_syn["release_probability"]
            else: # syn_mod does not have attribute for release probability so we approximate it by testing 1 release for entire simulation.
                p_test = self.random_state.uniform(low=0, high=1, size=1)
                if p_test < syn_params_this_syn["release_probability"]:
                    syn_params_this_syn["P_0"] = 1 #@TODO: when actually building synapses if syn_mod is not int2pyr or pyr2pyr then do not generate synapses with P_0 = 0. And skip assigning P_0 to the synapse object.
                else:
                    syn_params_this_syn["P_0"] = 0 #synapse is not releasing

            # sample an initial weight
            if initial_weight_distribution_is_partial:
                syn_params_this_syn["initW"] = initial_weight_distribution(size=1) # sample distribution
            else:
                syn_params_this_syn["initW"] = initial_weight_distribution # distribution is a constant

            # print(f"syn_params_this_syn: {syn_params_this_syn}")
            syn_params_this_syn["seg_id"] = segment['seg_id'].values[0]

            # pick out only the keys that contain 'gbar'
            gbar_params = {
                k: v
                for k, v in syn_params_this_syn.items()
                if 'gbar' in k
            }
            # add row to dataframe
            self.synapses = pd.concat((self.synapses, pd.DataFrame({
                'name': f"{name}_{_}",
                'modfile': syn_mod,
                # 'initW': syn_params_this_syn["initW"],
                # 'gmax': syn_params_this_syn["gmax"],
                # 'release_probability': syn_params_this_syn["release_probability"],
                'P_0': syn_params_this_syn["P_0"],
                'initW': syn_params_this_syn["initW"],
                'cell2cell_type': syn_params_this_syn["syn_params_choice"],

                'seg_id': syn_params_this_syn["seg_id"], 
                **gbar_params,
                # **syn_params_this_syn 
                # #TODO: (SHOULD BE DONE) use a string to indicate which syn_params to use instead of storing all of them in the DataFrame. Also will need to pull out the syn_params that were added to syn_params in this snippet (such as initW, location, release_probability.) and give them their own column.
            }, index=[0])), ignore_index=True)

def use_pssg(sim_dir: str, parameters, logger):
    # load parameters
    with open(os.path.join(sim_dir, "parameters.pickle"), 'rb') as file:
        parameters = pickle.load(file)

    # load segments
    segments = pd.read_csv(os.path.join(sim_dir, "segment_data.csv"))

    # create PreSimSynapseGenerator instance
    pssg = PreSimSynapseGenerator(segments=segments, parameters=parameters, logger=logger)
    pssg.generate_synapse_locations()
    pssg.synapses.to_csv(os.path.join(sim_dir, "synapses.csv"), index=False)
    # return pssg 


run_on_all_sims(sims_dir, use_pssg)

Generate synapse weights

In [18]:
# load synapse locations from synapses csv
synapses = pd.read_csv(os.path.join(sim_dir, "synapses.csv"))

# generate synapse weights, params, etc based on locations

# save to synapses csv in simulation folder

#@V-Marco TODO: Should we generate weights in the same loop as synapse generation, or have a separate section for it?
# is it faster to compute if we do not repeat the loop? 
# If weights are generated separately, then we have some flexibility with what synapses.csv we pass, and making hand alterations to locations before generated weights. 
# If we made a hand alteration later then we might also need to intelligently alter the weight.
# weights are currently generated in PreSimSynapseGenerator.generate_synapse_locations.

In [19]:
synapses

,name,modfile,P_0,initW,cell2cell_type,seg_id,gbar_ampa,gbar_nmda,gbar_gaba
0,exc_tuft_0,pyr2pyr,0.354387,1.548931,PN2PN,2023,0.00159,0.001868,NaN
1,exc_tuft_1,pyr2pyr,0.686850,1.683359,PN2PN,1638,0.00159,0.001868,NaN
2,exc_tuft_2,pyr2pyr,0.379372,1.416896,PN2PN,2140,0.00159,0.001868,NaN
3,exc_tuft_3,pyr2pyr,0.000000,1.761343,PN2PN,2085,0.00159,0.001868,NaN
4,exc_tuft_4,pyr2pyr,0.990291,1.251326,PN2PN,1720,0.00159,0.001868,NaN
...,...,...,...,...,...,...,...,...,...
18915,inh_perisomatic_174,int2pyr,0.864779,4.563609,PV2PN,11,NaN,NaN,0.05
18916,inh_perisomatic_175,int2pyr,0.901316,4.679414,PV2PN,435,NaN,NaN,0.05
18917,inh_perisomatic_176,int2pyr,0.966390,4.592213,PV2PN,222,NaN,NaN,0.05
18918,inh_perisomatic_177,int2pyr,0.830303,4.600967,PV2PN,717,NaN,NaN,0.05


In [20]:
segments = pd.read_csv(os.path.join(sim_dir, "segment_data.csv"))

In [21]:
segments

,section,idx_in_section_type,seg_half_seg_RA,L,length,seg,pseg,Section_L,Section_diam,Distance,...,pc_2,p1_0,p1_1,p1_2,r,dl_0,dl_1,dl_2,seg_id,sec_type_precise
0,soma,0,0.081276,23.169408,23.169408,L5PCtemplate[0].soma[0](0.5),None,23.169408,13.471518,0.000000,...,-50.250000,57.287731,19.065830,-50.250000,6.735759,23.124348,1.444304,0.000000,0,soma
1,dend,0,2.647135,4.840176,4.840176,L5PCtemplate[0].dend[0](0.1),L5PCtemplate[0].soma[0](0.5),24.200878,1.170000,2.420088,...,-50.027254,61.001504,21.408927,-49.444755,0.539452,4.591504,1.178928,0.805245,1,perisomatic
2,dend,0,2.250969,4.840176,4.840176,L5PCtemplate[0].dend[0](0.3),L5PCtemplate[0].dend[0](0.1),24.200878,1.170000,7.260263,...,-50.557475,64.964606,21.737526,-51.687077,0.585000,3.963102,0.328599,-2.242322,2,perisomatic
3,dend,0,2.250969,4.840176,4.840176,L5PCtemplate[0].dend[0](0.5),L5PCtemplate[0].dend[0](0.3),24.200878,1.170000,12.100439,...,-51.991813,68.301396,20.356904,-50.119401,0.585000,3.336790,-1.380622,1.567676,3,perisomatic
4,dend,0,2.250969,4.840176,4.840176,L5PCtemplate[0].dend[0](0.7),L5PCtemplate[0].dend[0](0.5),24.200878,1.170000,16.940615,...,-48.246989,70.726764,18.807201,-46.252588,0.585000,2.425368,-1.549704,3.866813,4,perisomatic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2625,axon,0,2.938245,4.615385,4.615385,L5PCtemplate[0].axon[0](0.653846),L5PCtemplate[0].axon[0](0.576923),60.000004,1.000000,39.230772,...,-50.250000,45.725559,59.882142,-50.250000,0.500000,0.000000,4.615385,0.000000,2625,axon
2626,axon,0,2.938245,4.615385,4.615385,L5PCtemplate[0].axon[0](0.730769),L5PCtemplate[0].axon[0](0.653846),60.000004,1.000000,43.846157,...,-50.250000,45.725559,64.497527,-50.250000,0.500000,0.000000,4.615385,0.000000,2626,axon
2627,axon,0,2.938245,4.615385,4.615385,L5PCtemplate[0].axon[0](0.807692),L5PCtemplate[0].axon[0](0.730769),60.000004,1.000000,48.461542,...,-50.250000,45.725559,69.112912,-50.250000,0.500000,0.000000,4.615385,0.000000,2627,axon
2628,axon,0,2.938245,4.615385,4.615385,L5PCtemplate[0].axon[0](0.884615),L5PCtemplate[0].axon[0](0.807692),60.000004,1.000000,53.076926,...,-50.250000,45.725559,73.728296,-50.250000,0.500000,0.000000,4.615385,0.000000,2628,axon


Generate Spike Trains

In [22]:
from Modules.spike_generator import PoissonTrainGenerator

def generate_spike_trains_for_synapses(segments:pd.DataFrame, synapses:pd.DataFrame)->None:

    synapses_with_seg_info = synapses.merge( # TODO: alternative could be used to save memory.
        segments, 
        on='seg_id', 
        how='left',               # carry along all synapses even if a seg_id is missing
        suffixes=('','_seg')      # e.g. if both have a 'length' column
    )

    columns = ['pc_0', 'pc_1', 'pc_2']

    # get the coordinates of the synapses
    synapse_coords = synapses_with_seg_info[columns].values

    random_state = np.random.RandomState(parameters.precell_spikes_seeds['inh'])
    np.random.seed(parameters.precell_spikes_seeds['inh'])

    synapses['spike_train'] = None                      # gives you dtype=object
    synapses['pc_mean_firing_rate'] = np.nan            # dtype float, which is fine

    for synapse_type in ['exc', 'inh']:
        for cluster_sec_type in getattr(parameters, f"{synapse_type}_syn_properties").keys():#['tuft']: # or ['trunk', 'distal_basal', 'distal_apic', 'nexus', 'oblique']: or getattr(parameters, f"{synapse_type}_syn_properties").keys():
            mean_firing_rate_distribution = getattr(parameters, f"{synapse_type}_syn_properties")[cluster_sec_type]['mean_firing_rate_distribution']

            # firing_rate_distribution = firing_rate_distribution['params']['mean'] if firing_rate_distribution['function'] is None else firing_rate_distribution['function']
            mean_firing_rate_distribution = partial(mean_firing_rate_distribution['function'], **mean_firing_rate_distribution['params'], size=1)

            # default to background firing rate
            fg_firing_rate_timecourse = np.ones(parameters.h_tstop) # background has no 'functional group' modulation
            synapses_this_sec_and_syn_type = (
                synapses['name'].str.contains(synapse_type, na=False) 
                & synapses['name'].str.contains(cluster_sec_type, na=False)
            )
            for idx, synapse_row in synapses[synapses_this_sec_and_syn_type].iterrows():
                pc_mean_firing_rate = mean_firing_rate_distribution(size=1)
                # synapse_row['pc_mean_firing_rate'] = pc_mean_firing_rate
                pc_firing_rate_timecourse = PoissonTrainGenerator.shift_mean_of_lambdas(lambdas=fg_firing_rate_timecourse, desired_mean=pc_mean_firing_rate)
                spike_train = PoissonTrainGenerator.generate_spike_train(lambdas=pc_firing_rate_timecourse, random_state=random_state)
                print(f"spike_train: {spike_train}")
                print(f"type(spike_train): {type(spike_train)}")
                # synapse_row['spike_train'] = spike_train.spike_times # TODO: check if this updates the synapse_row in the dataframe or if it needs to be added to the dataframe separately.

                # # update the synapse_row in the synapses dataframe
                # synapses.loc[synapses['name'] == synapse_row['name'], 'spike_train'] = [spike_train.spike_times] # wrap spike times in a list so pandas knows it's one cell
                # synapses.loc[synapses['name'] == synapse_row['name'], 'pc_mean_firing_rate'] = pc_mean_firing_rate 

                synapses.at[idx, 'spike_train'] = spike_train.spike_times
                synapses.at[idx, 'pc_mean_firing_rate'] = pc_mean_firing_rate

    synapses.to_csv(os.path.join(sim_dir, "synapses.csv"), index=False)


generate_spike_trains_for_synapses(segments, synapses)


#         # only consider synapses of this synapse type
#         synapse_ids_to_consider = synapses[synapses['name'].str.contains(synapse_type)]

# ## get cluster centers randomly
# # # get the mean and std of the coordinates
# # mean = np.mean(synapse_coords, axis=0)
# # std = np.std(synapse_coords, axis=0)
# # # get the range of the coordinates
# # range = np.max(synapse_coords, axis=0) - np.min(synapse_coords, axis=0)
# # # get 10 random cluster centers
# # cluster_centers = np.random.uniform(low=mean - 3*std, high=mean + 3*std, size=(10, 3)) # TODO: check (can be outside of mins and max, leading to error.)
# # cluster_centers = np.clip(cluster_centers, mean - 3*std, mean + 3*std)
# # get 10 random cluster centers by choosing among segments.
# cluster_centers = synapse_coords[random_state.choice(synapse_coords.shape[0], size=10, replace=False)]

# ## get synapse_ids for each cluster within bounds
# # get the coordinates of the synapses
# synapse_coords = synapses_with_seg_info[columns].values

# # use the distance of each synapse from the cluster center to determine if it belongs
# cluster_indices_by_cluster = []
# for cluster_center in cluster_centers:
#     distances = np.linalg.norm(synapse_coords - cluster_center, axis=1)#cluster_center, axis=1)
#     # get the indices of the synapses that are within 3 std of the cluster center
#     cluster_indices = np.where(distances < 100)[0]
#     # make sure the indices are unique across clusters
#     cluster_indices = np.unique(cluster_indices)

#     # right now cluster_indices tell the row of synapses_with_seg_info. Need to do the same, but
#     # only consider for cluster_indices the segments 
#     # of synapses_with_seg_info['sec_type_precise'] == cluster_sec_type
#     cluster_indices = np.where(synapses_with_seg_info['sec_type_precise'] == cluster_sec_type)[0][cluster_indices] #TODO: CHECK

#     # track for all clusters so we can deal with overlapping clusters
#     cluster_indices_by_cluster.append(cluster_indices)

# ## deal with overlapping clusters
# from collections import defaultdict
# # turn each cluster’s indices into a mutable set
# cluster_sets = [set(idxs) for idxs in cluster_indices_by_cluster]
# # build a map from each synapse‐index to the list of clusters it appears in
# idx_to_clusters = defaultdict(list)
# for cid, idxs in enumerate(cluster_sets):
#     for idx in idxs:
#         idx_to_clusters[idx].append(cid)
# # (optional) for reproducibility
# # np.random.seed(42)
# # for any index in >1 cluster, choose one cluster to keep it
# for idx, cids in idx_to_clusters.items():
#     if len(cids) > 1:
#         keep = np.random.choice(cids)
#         for cid in cids:
#             if cid != keep:
#                 cluster_sets[cid].remove(idx)

# # convert back to sorted numpy arrays (if you need arrays)
# cluster_indices_by_cluster = [
#     np.array(sorted(s)) for s in cluster_sets
# ]

# ## generate cluster spike trains
# # get cluster centers
# # get row indices that will be clustered for each cluster center (list of lists)
# # make sure that row indices are unique across clusters
# # generate spike trains for each cluster
# from Modules.spike_generator import PoissonTrainGenerator
# for cluster_indices in cluster_indices_by_cluster:
#     # generate FR profile for this cluster
#     firing_rates = PoissonTrainGenerator.generate_lambdas_from_pink_noise(
#         num = parameters.h_tstop,
#         random_state = random_state)
    
#     mean_fr = 

#     # generate spike train for each synapse
#     for synapse in synapses.iloc[cluster_indices]:
#         firing_rates_shifted = PoissonTrainGenerator.shift_mean_of_lambdas(firing_rates, desired_mean=mean_fr) 
#         spike_train = PoissonTrainGenerator.generate_spike_train(
#             lambdas = firing_rates_shifted, 
#             random_state = random_state)

# ## generate background spike train
# # get row indices that are not in clusters
# # generate spike trains for these synapses

spike_train: <Modules.spike_generator.SpikeTrain object at 0x7f30a2f76b60>
type(spike_train): <class 'Modules.spike_generator.SpikeTrain'>
spike_train: <Modules.spike_generator.SpikeTrain object at 0x7f30a2f778e0>
type(spike_train): <class 'Modules.spike_generator.SpikeTrain'>
spike_train: <Modules.spike_generator.SpikeTrain object at 0x7f30a2f751b0>
type(spike_train): <class 'Modules.spike_generator.SpikeTrain'>
spike_train: <Modules.spike_generator.SpikeTrain object at 0x7f30a2f76b60>
type(spike_train): <class 'Modules.spike_generator.SpikeTrain'>
spike_train: <Modules.spike_generator.SpikeTrain object at 0x7f30a2f778e0>
type(spike_train): <class 'Modules.spike_generator.SpikeTrain'>
spike_train: <Modules.spike_generator.SpikeTrain object at 0x7f30a2f751b0>
type(spike_train): <class 'Modules.spike_generator.SpikeTrain'>
spike_train: <Modules.spike_generator.SpikeTrain object at 0x7f30a2f76b60>
type(spike_train): <class 'Modules.spike_generator.SpikeTrain'>
spike_train: <Modules.spike

In [23]:
synapses

,name,modfile,P_0,initW,cell2cell_type,seg_id,gbar_ampa,gbar_nmda,gbar_gaba,spike_train,pc_mean_firing_rate
0,exc_tuft_0,pyr2pyr,0.354387,1.548931,PN2PN,2023,0.00159,0.001868,NaN,"[850, 1171, 1392, 1728, 2130, 2295, 2369, 2496...",3.041457
1,exc_tuft_1,pyr2pyr,0.686850,1.683359,PN2PN,1638,0.00159,0.001868,NaN,"[404, 917, 2572, 3099, 3419, 3602, 3654, 4983]",1.422990
2,exc_tuft_2,pyr2pyr,0.379372,1.416896,PN2PN,2140,0.00159,0.001868,NaN,"[4254, 4412]",0.806649
3,exc_tuft_3,pyr2pyr,0.000000,1.761343,PN2PN,2085,0.00159,0.001868,NaN,"[66, 71, 187, 307, 586, 1067, 1268, 1597, 1672...",4.006960
4,exc_tuft_4,pyr2pyr,0.990291,1.251326,PN2PN,1720,0.00159,0.001868,NaN,"[16, 141, 488, 739, 1164, 1260, 1290, 1456, 14...",5.585621
...,...,...,...,...,...,...,...,...,...,...,...
18915,inh_perisomatic_174,int2pyr,0.864779,4.563609,PV2PN,11,NaN,NaN,0.05,"[8, 131, 219, 242, 296, 345, 358, 431, 528, 56...",20.123716
18916,inh_perisomatic_175,int2pyr,0.901316,4.679414,PV2PN,435,NaN,NaN,0.05,"[16, 95, 185, 236, 282, 368, 382, 392, 399, 50...",16.726000
18917,inh_perisomatic_176,int2pyr,0.966390,4.592213,PV2PN,222,NaN,NaN,0.05,"[14, 32, 157, 179, 188, 242, 411, 610, 631, 66...",15.628864
18918,inh_perisomatic_177,int2pyr,0.830303,4.600967,PV2PN,717,NaN,NaN,0.05,"[21, 135, 227, 244, 252, 262, 267, 381, 417, 5...",15.806753


In [24]:
# USE cell_builder.assign_spikes and presynaptic.py for reference code. 
# @TODO: adapt presynaptic.py to use segments.csv instead of cell object. cell object is heavilty embedded in presynaptic.py module 

# generate functional groups from params

# generate presynaptic cells from functional groups and params

# generate spike trains for presynaptic cells from params

# load synapse locations from synapses csv

# cluster synapse locations into presynaptic cells

# assign synapses to presynaptic cells

# save spike trains and synapse assignments as spike_trains.csv to simulation folder


# #### new ####

# # generate 'background' for all. Then form clusters

# def generate_new_spike_trains(synapses: pd.DataFrame, segments:pd.DataFrame)-> pd.DataFrame: # TODO: make functions and class. started this way then realized better to not.
#     """
#     Generate spike trains for each synapse in the synapses DataFrame.
#     """
#     synapses_with_seg_info = synapses.merge( # TODO: alternative could be used to save memory.
#         segments, 
#         on='seg_id', 
#         how='left',               # carry along all synapses even if a seg_id is missing
#         suffixes=('','_seg')      # e.g. if both have a 'length' column
#     )

#     ## generate cluster spike trains
#     # get cluster center coordinates
#     centers_coords = [get_cluster_center_coords(synapses_with_seg_info)]
#     # get row indices that will be clustered for each cluster center (list of lists)
#     # make sure that row indices are unique across clusters
#     # generate spike trains for each cluster

#     ## generate background spike train
#     # get row indices that are not in clusters
#     # generate spike trains for these synapses

# def get_cluster_center_coords(synapses_with_seg_info: pd.DataFrame, num_centers:int = 10, columns:list = ['pc_0', 'pc_1', 'pc_2']) -> tuple:
#     """
#     Get the coordinates of the cluster center.
#     randomly pick coordiates within 3 std the range of the coordinates of the synapses #TODO: update to pick branches
#     """
#     # get the coordinates of the synapses
#     synapse_coords = synapses_with_seg_info[columns].values
#     # get the mean and std of the coordinates
#     mean = np.mean(synapse_coords, axis=0)
#     std = np.std(synapse_coords, axis=0)
#     # get the range of the coordinates
#     range = np.max(synapse_coords, axis=0) - np.min(synapse_coords, axis=0)

#     cluster_center = np.random.uniform(low=mean - 3*std, high=mean + 3*std, size=(num_centers,3))
#     # make sure the cluster center is within the range of the coordinates
#     cluster_center = np.clip(cluster_center, mean - 3*std, mean + 3*std)
#     # return the coordinates of the cluster center
#     return tuple(cluster_center)

# def get_cluster_row_indices(synapses_with_seg_info: pd.DataFrame, cluster_center: tuple, columns:list = ['pc_0', 'pc_1', 'pc_2'])-> list:
#     """
#     Get the row indices of the cluster provided the center of the cluster
#     """
#     # get the coordinates of the synapses
#     synapse_coords = synapses_with_seg_info[columns].values
#     # get the distance of each synapse from the cluster center
#     distances = np.linalg.norm(synapse_coords - cluster_center, axis=1)
#     # get the indices of the synapses that are within 3 std of the cluster center
#     cluster_indices = np.where(distances < 3*np.std(distances))[0]
#     # make sure the indices are unique across clusters
#     cluster_indices = np.unique(cluster_indices)
#     # return the indices of the synapses in the cluster
#     return cluster_indices.tolist()

# def generate_background_spike_trains():
#     pass

# def generate_cluster_spike_trains():
#     pass

Read synapse specifications and build cell with synapses

In [ ]:
import synapse

: 

In [ ]:
import synapse
from Modules.synapse import Synapse
from Modules.cell_model import CellModel
import numpy as np

def build_synapses_onto_cell_obj(sim_dir)->CellModel:
    def parse_array(s):
        # remove brackets, split on whitespace or commas…
        s = s.strip('[]')
        if not s:
            return np.array([])
        return np.fromstring(s, sep=' ')


    ## assumes already have parameters and sim_dir defined, even logger (use run_on_all_sims)

    # build synapses from synapses csv from simulation folder
    synapses = pd.read_csv(os.path.join(sim_dir, "synapses.csv"))
    # segments = pd.read_csv(os.path.join(sim_dir, "segment_data.csv"))

    syn_param_map = {cell2cell_type: getattr(synapse, f"{cell2cell_type}_syn_params") for cell2cell_type in np.unique(synapses.cell2cell_type)}

    # syn_param_map = {
    #     "PN2PN": synapse.PN2PN_syn_params,
    #     "CS2CP": synapse.CS2CP_syn_params,
    #     # …etc
    # }

    # set spike trains from spike_trains.csv from simulation folder

    logger = Logger(sim_dir) # create per‑sim logger (write info into "sims_dir/sim_dir/log.txt")

    # neuron_r = h.Random()
    # neuron_r = neuron_r.MCellRan4(parameters.neuron_random_state)

    logger.log(f"Building cell to generate segments.csv")
    cell_builder = CellBuilder(getattr(SkeletonCell, parameters.skeleton_cell_type), parameters, logger)
    cell, _ = cell_builder.build_cell()
    logger.log(f"Cell built successfully.")

    neuron_r = cell.neuron_r
    param_map = syn_param_map

    all_segments, seg_data = cell.get_segments(['all'])
    # build synapses
    # for synapse_idx, row in synapses.iterrows():
    #     print(all_segments)
    #     print(f"row: {row}")
    #     print(f"idx: {idx}")
    #     print(f"type(row): {type(row)}")

    #     print(row['seg_id'])
    #     #TODO: skip synapses where release probability = 0.
    #     #TODO: speed up building synapses.
    #     cell.synapses.append(Synapse(
    #         segment=all_segments[row['seg_id']], #@TODO: check that the seg from all_segments corresponds with the segment from segments.csv
    #         syn_mod=row['modfile'], 
    #         syn_params=getattr(synapse, f"{row['cell2cell_type']}_syn_params"),
    #         gmax = row['initW'],# gmax=gmax(size=1) if callable(gmax) else gmax,
    #         neuron_r=cell.neuron_r,
    #         name=row['name']))
        
    #     # print(f"row['spike_train]: {row['spike_train']}")
    #     # print(f"type(row['spike_train']): {type(row['spike_train'])}")
    #     # print(f"parsed spike train: {parse_array(row['spike_train'])}")
    #     cell.synapses[-1].set_spike_train(spike_train=np.array(row['spike_train']))

    # # a faster option that doesn't gathers all at once instead of individually.
    # for seg_id, modfile, cell_type, initW, name, spike_train in zip(
    #         synapses["seg_id"],
    #         synapses["modfile"],
    #         synapses["cell2cell_type"],
    #         synapses["initW"],
    #         synapses["name"],
    #         synapses["spike_train"],
    #             ):
    #     syn = Synapse(
    #         segment   = all_segments[seg_id],
    #         syn_mod   = modfile,
    #         syn_params= syn_param_map[cell_type],
    #         gmax      = initW,
    #         neuron_r  = cell.neuron_r,
    #         name      = name,
    #     )
    #     syn.set_spike_train(np.asarray(spike_train))  # faster than parse_array
    #     cell.synapses.append(syn)

    # an even faster option that uses list comprehension
    syn_list = [
        # unpack the namedtuple directly
        Synapse(
            segment    = all_segments[row.seg_id],
            syn_mod    = row.modfile,
            syn_params = param_map[row.cell2cell_type],
            gmax       = row.initW,
            neuron_r   = neuron_r,
            name       = row.name,
        )
        for row in synapses.itertuples(index=False)
    ]
    # 3. Now set all the spike trains in another pass
    for syn, train in zip(syn_list, synapses["spike_train"]):
        syn.set_spike_train(np.asarray(train))

    # 4. Finally attach them in one go
    cell.synapses.extend(syn_list)

cell = build_synapses_onto_cell_obj(sim_dir)

Removing duplicate coordinate at index 1 in section L5PCtemplate[0].apic[0]


Simulate